In [1]:
import os
import random
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.vim import VMamba, MambaConfig
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
class VimEncoder(nn.Module):
    """Wraps mambapy's VMamba -- bidirectional Vision Mamba
    encoder (per-block forward/backward SSM combination, matching Vim
    Algorithm)."""
    def __init__(self, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        config = MambaConfig(
            d_model=d_model, n_layers=n_layers, d_state=d_state,
            bidirectional=True, divide_output=True, pscan=True, use_cuda=False
        )
        self.encoder = VMamba(config)
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))

In [3]:
class ROIPatchEmbed3D(nn.Module):
    """Splits each 64^3 ROI into non-overlapping 8^3 patches -> tokens.
    Factorised positional embedding (ROI + depth + height + width) instead
    of one independent vector per patch position. Also returns a validity
    mask marking empty (background) patches for masked pooling."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32):
        super().__init__()
        if roi_size % patch_size != 0:
            raise ValueError("roi_size must be divisible by patch_size")
        self.n_rois = n_rois
        self.patch_size = patch_size
        self.grid_size = roi_size // patch_size
        self.patches_per_roi = self.grid_size ** 3
        self.d_model = d_model

        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.roi_embed = nn.Embedding(n_rois, d_model)
        self.depth_embed = nn.Embedding(self.grid_size, d_model)
        self.height_embed = nn.Embedding(self.grid_size, d_model)
        self.width_embed = nn.Embedding(self.grid_size, d_model)

        # Scale down from nn.Embedding's default N(0,1) init -- matches the
        # *0.02 scaling used for the old full positional table, so the
        # positional signal doesn't dominate the patch content signal early
        # in training.
        with torch.no_grad():
            self.roi_embed.weight.mul_(0.02)
            self.depth_embed.weight.mul_(0.02)
            self.height_embed.weight.mul_(0.02)
            self.width_embed.weight.mul_(0.02)

        d, h, w = torch.meshgrid(
            torch.arange(self.grid_size), torch.arange(self.grid_size),
            torch.arange(self.grid_size), indexing="ij"
        )
        coordinates = torch.stack([d, h, w], dim=-1).reshape(-1, 3)
        self.register_buffer("coordinates", coordinates, persistent=False)

    def forward(self, rois):
        batch_size, n_rois = rois.shape[:2]
        x = rois.reshape(batch_size * n_rois, 1, rois.shape[-3], rois.shape[-2], rois.shape[-1])

        tokens = self.patch_conv(x)
        tokens = tokens.flatten(2).transpose(1, 2)
        tokens = tokens.reshape(batch_size, n_rois, self.patches_per_roi, self.d_model)

        coords = self.coordinates
        spatial_position = (self.depth_embed(coords[:, 0]) +
                             self.height_embed(coords[:, 1]) +
                             self.width_embed(coords[:, 2]))
        roi_position = self.roi_embed.weight[:, None, :]

        tokens = tokens + spatial_position[None, None, :, :]
        tokens = tokens + roi_position[None, :, :, :]

        occupancy = F.max_pool3d((x.abs() > 1e-6).float(), kernel_size=self.patch_size, stride=self.patch_size)
        valid_patch = occupancy.flatten(1).bool().reshape(batch_size, n_rois, self.patches_per_roi)

        return tokens.reshape(batch_size, -1, self.d_model), valid_patch.reshape(batch_size, -1)


class VisionMambaBranch(nn.Module):
    """Factorised patch embed -> real Vim (VMamba) -> masked pooling.
    Returns BOTH the overall pooled vector (for classification) AND
    per-ROI pooled vectors (for later explainability -- ROI-level
    contribution analysis, feature importance ranking)."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        self.n_rois = n_rois
        self.patch_embed = ROIPatchEmbed3D(n_rois, roi_size, patch_size, d_model)
        self.vim = VimEncoder(d_model, n_layers, d_state)

    def forward(self, rois):
        tokens, valid_mask = self.patch_embed(rois)
        tokens = self.vim(tokens)  # (B, n_rois*patches_per_roi, d_model)

        # Overall pooled vector (used for classification)
        weights = valid_mask.unsqueeze(-1).to(tokens.dtype)
        pooled = (tokens * weights).sum(dim=1) / weights.sum(dim=1).clamp_min(1.0)

        # Per-ROI pooled vectors (saved for later explainability, not used in loss)
        B = tokens.shape[0]
        tokens_by_roi = tokens.reshape(B, self.n_rois, -1, tokens.shape[-1])
        mask_by_roi = valid_mask.reshape(B, self.n_rois, -1, 1).to(tokens.dtype)
        roi_embeddings = (tokens_by_roi * mask_by_roi).sum(dim=2) / mask_by_roi.sum(dim=2).clamp_min(1.0)

        return pooled, roi_embeddings  # (B, d_model), (B, n_rois, d_model)


class VisionMambaModel(nn.Module):
    """Single-modality model -- use for MRI-only or PET-only."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, rois, return_roi_embeddings=False):
        pooled, roi_embeddings = self.branch(rois)
        logits = self.classifier(self.dropout(pooled))
        if return_roi_embeddings:
            return logits, roi_embeddings
        return logits


class MultimodalVisionMambaModel(nn.Module):
    """Late fusion -- separate MRI/PET branches, concatenated before classifier."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.mri_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.pet_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)

    def forward(self, mri_rois, pet_rois, return_roi_embeddings=False):
        mri_pooled, mri_roi_emb = self.mri_branch(mri_rois)
        pet_pooled, pet_roi_emb = self.pet_branch(pet_rois)
        fused = torch.cat([mri_pooled, pet_pooled], dim=1)
        logits = self.classifier(self.dropout(fused))
        if return_roi_embeddings:
            return logits, mri_roi_emb, pet_roi_emb
        return logits

In [4]:
COHORT_CSV    = "D:/mamba_model/thesis_cohort_final.csv"
MRI_CACHE_AUG = "D:/mamba_model/preprocessed_cache_roi64_aug"
PET_CACHE_AUG = "D:/mamba_model/preprocessed_cache_pet_aug"
CKPT_DIR      = "D:/mamba_model/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

df = pd.read_csv(COHORT_CSV)

required_columns = {"subject_id", "mri_session", "outcome_label"}
missing = required_columns.difference(df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")
if not df["mri_session"].is_unique:
    raise ValueError("mri_session is not unique")
if df["subject_id"].nunique() != len(df):
    raise ValueError("Multiple rows exist for at least one subject -- split must be subject-level.")

print("Rows:", len(df))
print("Unique subjects:", df["subject_id"].nunique())
print(df["outcome_label"].value_counts().sort_index())

sessions = df["mri_session"].values
labels   = df["outcome_label"].values

X_tv, X_test, y_tv, y_test = train_test_split(
    sessions, labels, test_size=0.2, random_state=42, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv
)
session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))

print(f"\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Rows: 210
Unique subjects: 210
outcome_label
0    103
1    107
Name: count, dtype: int64

Train: 126 | Val: 42 | Test: 42


In [5]:
class ROIDataset(Dataset):
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples = []
        self.cache_dir = cache_dir
        for session_id, label in zip(sessions, labels):
            key = session_id if is_mri else session_to_subject[session_id]
            self.samples.append((key, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((key, label, f"aug{seed}"))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        key, label, version = self.samples[idx]
        path = f"{self.cache_dir}/{key}_{version}.npy"
        rois = np.load(path, mmap_mode="r")
        if rois.shape != (6, 64, 64, 64):
            raise ValueError(f"Unexpected shape {rois.shape}: {path}")
        rois = np.asarray(rois, dtype=np.float32)
        return torch.from_numpy(rois).unsqueeze(1), torch.tensor(label, dtype=torch.long), key


class MultimodalROIDataset(Dataset):
    def __init__(self, sessions, labels, mri_cache_dir, pet_cache_dir, is_train=False):
        self.samples = []
        self.mri_cache_dir = mri_cache_dir
        self.pet_cache_dir = pet_cache_dir
        for session_id, label in zip(sessions, labels):
            subject_id = session_to_subject[session_id]
            self.samples.append((session_id, subject_id, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((session_id, subject_id, label, f"aug{seed}"))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        mri_key, pet_key, label, version = self.samples[idx]
        mri_rois = np.asarray(np.load(f"{self.mri_cache_dir}/{mri_key}_{version}.npy", mmap_mode="r"), dtype=np.float32)
        pet_rois = np.asarray(np.load(f"{self.pet_cache_dir}/{pet_key}_{version}.npy", mmap_mode="r"), dtype=np.float32)
        return (torch.from_numpy(mri_rois).unsqueeze(1), torch.from_numpy(pet_rois).unsqueeze(1),
                torch.tensor(label, dtype=torch.long), mri_key)

In [6]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for rois, labels, _ in loader:
        rois, labels = rois.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(rois)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for rois, labels, _ in loader:
            rois, labels = rois.to(device), labels.to(device)
            outputs = model(rois)
            total_loss += criterion(outputs, labels).item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    tpr = recall_score(all_labels, all_preds, zero_division=0)
    tnr = specificity_score(all_labels, all_preds)
    return avg_loss, acc, tpr, tnr


def train_epoch_mm(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for mri_rois, pet_rois, labels, _ in loader:
        mri_rois, pet_rois, labels = mri_rois.to(device), pet_rois.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(mri_rois, pet_rois)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_mm(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for mri_rois, pet_rois, labels, _ in loader:
            mri_rois, pet_rois, labels = mri_rois.to(device), pet_rois.to(device), labels.to(device)
            outputs = model(mri_rois, pet_rois)
            total_loss += criterion(outputs, labels).item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    tpr = recall_score(all_labels, all_preds, zero_division=0)
    tnr = specificity_score(all_labels, all_preds)
    return avg_loss, acc, tpr, tnr

In [7]:
def measure_inference_time(model, loader, device, is_multimodal, n_batches_to_time=20):
    model.eval()
    times = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches_to_time:
                break
            if is_multimodal:
                mri_rois, pet_rois, labels, _ = batch
                mri_rois, pet_rois = mri_rois.to(device), pet_rois.to(device)
                batch_size = mri_rois.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time()
                _ = model(mri_rois, pet_rois)
            else:
                rois, labels, _ = batch
                rois = rois.to(device)
                batch_size = rois.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time()
                _ = model(rois)
            if device.type == 'cuda': torch.cuda.synchronize()
            times.append((time.time() - t0) / batch_size)
    return np.mean(times), np.std(times)


def try_compute_flops(model, loader, device, is_multimodal):
    try:
        model.eval()
        sample_batch = next(iter(loader))
        with torch.no_grad():
            if is_multimodal:
                mri_rois, pet_rois, labels, _ = sample_batch
                sample_input = (mri_rois[:1].to(device), pet_rois[:1].to(device))
                macs, params = profile(model, inputs=sample_input, verbose=False)
            else:
                rois, labels, _ = sample_batch
                sample_input = rois[:1].to(device)
                macs, params = profile(model, inputs=(sample_input,), verbose=False)
        return macs * 2
    except Exception as e:
        print(f"  (FLOPs estimation failed: {e})")
        return None


def extract_roi_embeddings(model, loader, device, is_multimodal):
    """Runs the test set once more to collect per-ROI embeddings + labels +
    subject keys for later explainability analysis (ROI contribution
    ranking, probe classifiers) -- no retraining needed later."""
    model.eval()
    records = []
    with torch.no_grad():
        if is_multimodal:
            for mri_rois, pet_rois, labels, keys in loader:
                mri_rois, pet_rois = mri_rois.to(device), pet_rois.to(device)
                logits, mri_roi_emb, pet_roi_emb = model(mri_rois, pet_rois, return_roi_embeddings=True)
                for i, key in enumerate(keys):
                    records.append({
                        "key": key, "label": labels[i].item(),
                        "mri_roi_embeddings": mri_roi_emb[i].cpu().numpy(),
                        "pet_roi_embeddings": pet_roi_emb[i].cpu().numpy(),
                    })
        else:
            for rois, labels, keys in loader:
                rois = rois.to(device)
                logits, roi_emb = model(rois, return_roi_embeddings=True)
                for i, key in enumerate(keys):
                    records.append({
                        "key": key, "label": labels[i].item(),
                        "roi_embeddings": roi_emb[i].cpu().numpy(),
                    })
    return records


def run_one_seed(seed, model_class, train_loader, val_loader, test_loader,
                  is_multimodal, save_prefix, max_epochs=101, patience=15):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = model_class(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

    train_fn = train_epoch_mm if is_multimodal else train_epoch
    eval_fn = evaluate_mm if is_multimodal else evaluate

    best_val_loss = float("inf")
    no_improvement = 0
    best_epoch = 0
    save_path = f"{CKPT_DIR}/{save_prefix}_seed{seed}.pt"
    total_train_time = 0

    print(f"\n--- Seed {seed} ---")
    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'Val TPR':>8} | {'Val TNR':>8} | {'Time':>6}")
    print("-" * 70)

    for epoch in range(1, max_epochs):
        t0 = time.time()
        train_loss = train_fn(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_tpr, val_tnr = eval_fn(model, val_loader, criterion, device)
        scheduler.step(val_loss)
        epoch_time = time.time() - t0
        total_train_time += epoch_time

        print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | "
              f"{val_acc:>8.4f} | {val_tpr:>8.4f} | {val_tnr:>8.4f} | {epoch_time:>5.1f}s")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            no_improvement = 0
            torch.save(model.state_dict(), save_path)
        else:
            no_improvement += 1
            if no_improvement >= patience:
                print(f"Early stopping at epoch {epoch}. Best: {best_epoch}")
                break

    model.load_state_dict(torch.load(save_path, weights_only=True))
    test_loss, test_acc, test_tpr, test_tnr = eval_fn(model, test_loader, criterion, device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    inf_time_mean, inf_time_std = measure_inference_time(model, test_loader, device, is_multimodal)
    flops = try_compute_flops(model, test_loader, device, is_multimodal)
    flops_str = f"{flops/1e9:.2f}GFLOPs" if flops else "N/A"

    roi_records = extract_roi_embeddings(model, test_loader, device, is_multimodal)
    roi_save_path = f"{CKPT_DIR}/{save_prefix}_seed{seed}_roi_embeddings.npy"
    np.save(roi_save_path, roi_records, allow_pickle=True)

    print(f"\n  >>> Seed {seed} TEST: Acc={test_acc*100:.1f}% | TPR={test_tpr*100:.1f}% | TNR={test_tnr*100:.1f}% | "
          f"train_time={total_train_time/60:.1f}min | inf={inf_time_mean*1000:.2f}ms | {flops_str}")
    print(f"      ROI embeddings saved: {roi_save_path}")

    return {"seed": seed, "acc": test_acc, "tpr": test_tpr, "tnr": test_tnr,
            "best_epoch": best_epoch, "train_time_sec": total_train_time,
            "n_params": n_params, "inf_time_ms": inf_time_mean * 1000, "flops": flops}

In [8]:
mri_train_dataset = ROIDataset(X_train, y_train, MRI_CACHE_AUG, is_mri=True, is_train=True)
mri_val_dataset   = ROIDataset(X_val,   y_val,   MRI_CACHE_AUG, is_mri=True, is_train=False)
mri_test_dataset  = ROIDataset(X_test,  y_test,  MRI_CACHE_AUG, is_mri=True, is_train=False)

BATCH_SIZE = 4
mri_train_loader = DataLoader(mri_train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
mri_val_loader   = DataLoader(mri_val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
mri_test_loader  = DataLoader(mri_test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("=== MRI-ONLY: 3-seed run (real VMamba, complete model) ===")
mri_seed_results = []
for seed in [1, 7, 123]:
    result = run_one_seed(seed, VisionMambaModel, mri_train_loader, mri_val_loader, mri_test_loader,
                           is_multimodal=False, save_prefix="vim_final_mri_d32")
    mri_seed_results.append(result)

=== MRI-ONLY: 3-seed run (real VMamba, complete model) ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------


C:\Users\sammy\AppData\Local\Temp\ipykernel_8396\4251363853.py:22: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_numpy.cpp:212.)
  return torch.from_numpy(rois).unsqueeze(1), torch.tensor(label, dtype=torch.long), key


     1 |     0.7053 |     0.6948 |   0.4762 |   0.9048 |   0.0476 |   8.7s
     2 |     0.6982 |     0.6919 |   0.5476 |   0.4762 |   0.6190 |   8.3s
     3 |     0.6980 |     0.6897 |   0.5476 |   0.2857 |   0.8095 |   8.4s
     4 |     0.6889 |     0.6887 |   0.4762 |   0.9524 |   0.0000 |   8.4s
     5 |     0.6904 |     0.6868 |   0.5476 |   0.9048 |   0.1905 |   8.5s
     6 |     0.6890 |     0.6850 |   0.5714 |   0.3333 |   0.8095 |   8.4s
     7 |     0.6877 |     0.6846 |   0.5952 |   0.6667 |   0.5238 |   8.3s
     8 |     0.6874 |     0.6832 |   0.6190 |   0.6667 |   0.5714 |   8.4s
     9 |     0.6869 |     0.6818 |   0.5714 |   0.3333 |   0.8095 |   8.3s
    10 |     0.6822 |     0.6800 |   0.5714 |   0.3333 |   0.8095 |   8.3s
    11 |     0.6856 |     0.6800 |   0.5952 |   0.2381 |   0.9524 |   8.3s
    12 |     0.6804 |     0.6777 |   0.6190 |   0.9048 |   0.3333 |   8.4s
    13 |     0.6797 |     0.6758 |   0.6667 |   0.8095 |   0.5238 |   8.5s
    14 |     0.6745 |    

In [9]:
pet_train_dataset = ROIDataset(X_train, y_train, PET_CACHE_AUG, is_mri=False, is_train=True)
pet_val_dataset   = ROIDataset(X_val,   y_val,   PET_CACHE_AUG, is_mri=False, is_train=False)
pet_test_dataset  = ROIDataset(X_test,  y_test,  PET_CACHE_AUG, is_mri=False, is_train=False)

pet_train_loader = DataLoader(pet_train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
pet_val_loader   = DataLoader(pet_val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
pet_test_loader  = DataLoader(pet_test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("=== PET-ONLY: 3-seed run ===")
pet_seed_results = []
for seed in [1, 7, 123]:
    result = run_one_seed(seed, VisionMambaModel, pet_train_loader, pet_val_loader, pet_test_loader,
                           is_multimodal=False, save_prefix="vim_final_pet_d32")
    pet_seed_results.append(result)

=== PET-ONLY: 3-seed run ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7060 |     0.6879 |   0.5714 |   0.3333 |   0.8095 |  15.6s
     2 |     0.6961 |     0.6854 |   0.5476 |   0.1905 |   0.9048 |   8.0s
     3 |     0.6949 |     0.6846 |   0.5714 |   0.2381 |   0.9048 |   7.9s
     4 |     0.6886 |     0.6840 |   0.5476 |   0.5714 |   0.5238 |   8.2s
     5 |     0.6889 |     0.6836 |   0.5238 |   0.4286 |   0.6190 |   8.2s
     6 |     0.6864 |     0.6830 |   0.6190 |   0.3333 |   0.9048 |   8.1s
     7 |     0.6865 |     0.6815 |   0.5714 |   0.4762 |   0.6667 |   8.2s
     8 |     0.6840 |     0.6816 |   0.5476 |   0.4762 |   0.6190 |   8.1s
     9 |     0.6841 |     0.6809 |   0.6190 |   0.4762 |   0.7619 |   8.6s
    10 |     0.6804 |     0.6805 |   0.5952 |   0.3810 |   0.8095 |   8.8s
    11 |     0.6825 |     0.6801 |   0.6190 |   0.3333 |   

In [10]:
mm_train_dataset = MultimodalROIDataset(X_train, y_train, MRI_CACHE_AUG, PET_CACHE_AUG, is_train=True)
mm_val_dataset   = MultimodalROIDataset(X_val,   y_val,   MRI_CACHE_AUG, PET_CACHE_AUG, is_train=False)
mm_test_dataset  = MultimodalROIDataset(X_test,  y_test,  MRI_CACHE_AUG, PET_CACHE_AUG, is_train=False)

mm_train_loader = DataLoader(mm_train_dataset, batch_size=4, shuffle=True,  num_workers=0)
mm_val_loader   = DataLoader(mm_val_dataset,   batch_size=4, shuffle=False, num_workers=0)
mm_test_loader  = DataLoader(mm_test_dataset,  batch_size=4, shuffle=False, num_workers=0)

print("=== MULTIMODAL: 3-seed run ===")
mm_seed_results = []
for seed in [1, 7, 123]:
    result = run_one_seed(seed, MultimodalVisionMambaModel, mm_train_loader, mm_val_loader, mm_test_loader,
                           is_multimodal=True, save_prefix="vim_final_mm_d32")
    mm_seed_results.append(result)

=== MULTIMODAL: 3-seed run ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.6967 |     0.6901 |   0.5000 |   1.0000 |   0.0000 |  84.0s
     2 |     0.6996 |     0.6955 |   0.5000 |   1.0000 |   0.0000 |  18.0s
     3 |     0.6945 |     0.6888 |   0.5000 |   1.0000 |   0.0000 |  18.3s
     4 |     0.6904 |     0.6831 |   0.5952 |   0.2381 |   0.9524 |  18.2s
     5 |     0.6848 |     0.6819 |   0.5476 |   0.4286 |   0.6667 |  18.2s
     6 |     0.6832 |     0.6817 |   0.5714 |   0.1905 |   0.9524 |  17.8s
     7 |     0.6894 |     0.6808 |   0.5476 |   0.7143 |   0.3810 |  17.7s
     8 |     0.6807 |     0.6801 |   0.5476 |   0.7619 |   0.3333 |  18.0s
     9 |     0.6849 |     0.6778 |   0.5714 |   0.6190 |   0.5238 |  18.0s
    10 |     0.6743 |     0.6755 |   0.5476 |   0.4762 |   0.6190 |  18.0s
    11 |     0.6760 |     0.6738 |   0.5714 |   0.4286 | 

KeyboardInterrupt: 

In [ ]:
def summarize(results, name):
    accs = [r["acc"] for r in results]
    tprs = [r["tpr"] for r in results]
    tnrs = [r["tnr"] for r in results]
    times = [r["train_time_sec"] for r in results]
    infs = [r["inf_time_ms"] for r in results]
    flops_vals = [r["flops"] for r in results if r["flops"] is not None]

    print(f"\n{name} SUMMARY (3 seeds)")
    print(f"  Acc: {np.mean(accs)*100:.1f}±{np.std(accs)*100:.1f}%")
    print(f"  TPR: {np.mean(tprs)*100:.1f}±{np.std(tprs)*100:.1f}%")
    print(f"  TNR: {np.mean(tnrs)*100:.1f}±{np.std(tnrs)*100:.1f}%")
    print(f"  Params: {results[0]['n_params']:,}")
    print(f"  Train time: {np.mean(times)/60:.1f}±{np.std(times)/60:.1f} min/seed")
    print(f"  Inference: {np.mean(infs):.2f}±{np.std(infs):.2f} ms/scan")
    if flops_vals:
        print(f"  FLOPs (approx): {np.mean(flops_vals)/1e9:.2f} GFLOPs")

    return {"acc_mean": np.mean(accs), "acc_std": np.std(accs), "tpr_mean": np.mean(tprs),
            "tpr_std": np.std(tprs), "tnr_mean": np.mean(tnrs), "tnr_std": np.std(tnrs),
            "n_params": results[0]["n_params"], "train_time_min": np.mean(times)/60,
            "inf_time_ms": np.mean(infs), "flops_gflops": np.mean(flops_vals)/1e9 if flops_vals else None}


mri_summary = summarize(mri_seed_results, "MRI-ONLY")
pet_summary = summarize(pet_seed_results, "PET-ONLY")
mm_summary  = summarize(mm_seed_results, "MULTIMODAL")

print(f"\n\n{'='*100}")
print(f"{'Model':<20} | {'Accuracy':>13} | {'TPR':>13} | {'TNR':>13} | {'Params':>9} | {'Train':>8} | {'Inference':>10} | {'FLOPs':>10}")
print("-" * 100)
print(f"{'MNA-net (Vo et al.)':<20} | {'82.9%':>13} | {'85.7%':>13} | {'80.0%':>13} | {'--':>9} | {'--':>8} | {'--':>10} | {'--':>10}")
for name, s in [("MRI-only", mri_summary), ("PET-only", pet_summary), ("Multimodal", mm_summary)]:
    flops_str = f"{s['flops_gflops']:.2f}G" if s['flops_gflops'] else "N/A"
    print(f"{name:<20} | {s['acc_mean']*100:>5.1f}±{s['acc_std']*100:<5.1f}% | "
          f"{s['tpr_mean']*100:>5.1f}±{s['tpr_std']*100:<5.1f}% | "
          f"{s['tnr_mean']*100:>5.1f}±{s['tnr_std']*100:<5.1f}% | "
          f"{s['n_params']:>9,} | {s['train_time_min']:>6.1f}m | {s['inf_time_ms']:>8.2f}ms | {flops_str:>10}")

print(f"\nROI embeddings saved per seed in {CKPT_DIR}/*_roi_embeddings.npy for later explainability analysis.")